In [ ]:
"""
Downloads a LANDFIRE Landscape file (multi-band geotiff) from the LFPS API
Optionally, applies an edit rule for lodgepole pine adjustments, etc.

Maxwell.Cook@colostate.edu
"""

# standard imports
import os, sys
import geopandas as gpd

# custom functions
sys.path.append(os.getcwd()) # add code folder to system path
from code.Python.archive.__functions import *  # imports all custom functions

# directories
# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

print("Ready to go !")

In [ ]:
# load the area of interest
fp = os.path.join(projdir, 'data/spatial/boundaries/NCFC/NCFC_QRA_extent.gpkg')
aoi = gpd.read_file(fp).to_crs(4326) # WGS for lat/lon
# gather the URL-encoded bounding box
bbox = aoi.total_bounds # list
bbox_str = f"{bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}" # string format
print(bbox_str)

### Access LANDFIRE fuelscape using the LFPS API

The LANDFIRE Product Services API can be used to request and download a fuelscape (multi-band geotiff) including options for queries and modifications. This can be used to download fuelscapes with the lodgepole pine adjustment performed in the request.

In [ ]:
# --- Make the LFPS request (see __functions.py)


In [ ]:
# submit a job to the LFPS to gather the required landscape inputs (multi-band GeoTiff)
lfps_url = "https://lfps.usgs.gov/api/job/submit" # URL for job submission
lf_v = '230' # which LANDFIRE product vintage? 200CC_20 = Forest Canopy Cover ca. 2020
# define other parameters for the URL request
# required params include layer list, email, bounding box, and projection
# see https://lfps.usgs.gov/LFProductsServiceUserGuide.pdf
layer_list = f"ELEV2020;SLPD2020;ASP2020;{lf_v}FBFM40;{lf_v}CC;{lf_v}CH;{lf_v}CBH;{lf_v}CBD;{lf_v}EVT"
email = "Maxwell.Cook@colostate.edu"
# Optional params include resample resolution, output projection, and edit rule
# edit rule can be used to modify the inputs (e.g., changing FBFM40 fuel model based on a condition)
# e.g., here is the required lodgepole adjustment rule defined by CFRI for other projects:
lodgepole_adjust = {
    "edit": [
        {
            # change surface fuel model from 181 or 183 to 185
            "condition": [
                {"product": f"{lf_v}EVT", "operator": "EQ", "value": 7050},
                {"product": f"{lf_v}FBFM40", "operator": "EQ", "value": 181},
            ],
            "change": [
                {"product": f"{lf_v}FBFM40", "operator": "ST", "value": 185}
            ]
        },
        {
            # change surface fuel model from 181 or 183 to 185
            "condition": [
                {"product": f"{lf_v}EVT", "operator": "EQ", "value": 7050},
                {"product": f"{lf_v}FBFM40", "operator": "EQ", "value": 183}
            ],
            "change": [
                {"product": f"{lf_v}FBFM40", "operator": "ST", "value": 185}
            ]
        },
        {
            # adjust the CBH for lodgepole pixel by 70%
            "condition": [
                {"product": f"{lf_v}EVT", "operator": "EQ", "value": 7050},
            ],
            "change": [
                {"product": f"{lf_v}CBH", "operator": "MB", "value": 0.7}
            ]
        },
    ]
}

# define the request parameters
params = {
    "Layer_List": layer_list,
    "Area_of_Interest": bbox_str,
    "Output_Projection": proj_crs,
    "Email": email,
    # edit rule for adjusting lodgepole fuel model
    "Edit_Rule": json.dumps(lodgepole_adjust) # format the edit rule in the correct way using json dumps
}

# make the request
r = requests.get(lfps_url, params=params)
print(f"Status Code: {r.status_code}")
print(f"Response: {r.text}")

In [ ]:
# check job status for download URL
# get the job ID from response
job_data = r.json()
job_id = job_data["jobId"]
# format the job status url
status_url = f"https://lfps.usgs.gov/api/job/status?JobId={job_id}"

# check the status until complete.
while True:
    try:
        resp = requests.get(status_url)
        data = resp.json()
        status = data.get("status", "Unknown")

        print(f"[{time.ctime()}] Job Status: {status}")

        if status == "Succeeded":
            print("Job completed successfully!")
            print(data)
            break
        elif status in ["Failed", "Cancelled"]:
            messages = data.get("messages", [])
            raise RuntimeError(f"LFPS job failed. Status: {status}. Messages: {messages}")
        else:
            # Still in progress
            time.sleep(60)

    # try to catch the exceptions
    except requests.exceptions.RequestException as req_err:
        print(f"[{time.ctime()}] Request error: {req_err} — retrying in 60 sec")
        time.sleep(60)
    except ValueError as json_err:
        print(f"[{time.ctime()}] JSON decode error: {json_err} — retrying in 60 sec")
        time.sleep(60)
    except RuntimeError as critical_err:
        # Allow failure to break the loop and optionally handle or re-raise
        print(f"[{time.ctime()}] Critical error: {critical_err}")
        raise

## Download and extract the LFPS output

Once the file is downloaded and extracted, remove the EVT band as this is no longer needed for FlamMap runs. Save the final multi-band GeoTIFF as the baseline fuelscape.

In [ ]:
# download the landscape file to a local directory
download_url = data["outputFile"] # the download link from the LFPS request

# specify the output directory and file name
aoi = 'NCFC'
output_dir = os.path.join(projdir, fr"data\spatial\fuelscape\{aoi}\LFPS")
os.makedirs(output_dir, exist_ok=True)

# clear the directory before extracting the zip file
for file in glob.glob(os.path.join(output_dir, "*")):
    if os.path.isfile(file):
        os.remove(file)

# download & extract the zip file
r = requests.get(download_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(output_dir)
z.close()

print("\nDownloaded and extracted to:", output_dir, "\n")

# drop the EVT band and save a file for FlamMap
# clean up the LFPS download file
# load in and check the bands/metadata
lcp_fp = list_files(output_dir, "*.tif", recursive=True)[0]
with rxr.open_rasterio(lcp_fp, masked=True) as lcp_tif:
    # remove the EVT band, which is not needed for FlamMap but was needed to adjust lodgepole fuels
    # reassign band names via the 'long_name' attribute
    lcp_fm = lcp_tif.sel(band=slice(1, 8)).astype("int16")
    lcp_fm.attrs["long_name"] = (
        'ELEV', 'SLP', 'ASP', 'FBFM40',
        'CC', 'CH', 'CBH', 'CBD'
    )
    # ensure CRS is defined
    if lcp_fm.rio.crs is None:
        lcp_fm.rio.write_crs(lcp_tif.rio.crs, inplace=True)

# check on the tiff file
print("Baseline fuelscape:")
print("Shape:", lcp_fm.shape)  # should be (8, y, x)
print("Data type:", lcp_fm.dtype)
print("Band names:", lcp_fm.attrs.get("long_name", "unknown"))

shutil.rmtree(output_dir) # remove the LFPS download folder

# (optional) crop to the AOI
crop = False
if crop is True:
    # re-load the area of interest
    fp = os.path.join(projdir, 'data/spatial/boundaries/NCFC/NCFC_QRA_extent.gpkg')
    aoi = gpd.read_file(fp)
    aoi['geometry'] = aoi.geometry.buffer(90)
    aoi = aoi.to_crs(lcp_fm.rio.crs) # match crs
    lcp_fm = lcp_fm.rio.clip(aoi)

# save the file out.
output_dir = os.path.join(projdir, fr"data\spatial\fuelscape\{aoi}\baseline")
os.makedirs(output_dir, exist_ok=True)
out_fp = os.path.join(output_dir, "fuelscape_baseline.tif")
lcp_fm.rio.to_raster(out_fp, compress='zstd', zstd_level=9)
print(f"\nSaved to: {out_fp}\n")

In [ ]:
# --- Stamp in historic treatments pre-QWRA (2021)
